In [ ]:
import numpy as np
import do_mpc
from utils.Dataclasses import Study
from casadi import vertcat
from dataclasses import asdict
from pprint import pprint
from tqdm import tqdm

c:\Users\janlc\anaconda3\envs\GeothermalSorption\Lib\site-packages\do_mpc\sysid\__init__.py:15: UserWarning: The ONNX feature is not available. Please install the full version of do-mpc to access this feature.
  warnings.warn('The ONNX feature is not available. Please install the full version of do-mpc to access this feature.')
c:\Users\janlc\anaconda3\envs\GeothermalSorption\Lib\site-packages\do_mpc\opcua\__init__.py:14: UserWarning: The opcua feature is not available. Please install the full version of do-mpc to access this feature.
  warnings.warn('The opcua feature is not available. Please install the full version of do-mpc to access this feature.')
c:\Users\janlc\anaconda3\envs\GeothermalSorption\Lib\site-packages\do_mpc\approximateMPC\__init__.py:13: UserWarning: The approximateMPC feature requires PyTorch, which is not installed by default. Please install the full version of do-mpc to use this feature: pip install do_mpc[full]
  warnings.warn('The approximateMPC feature requires

In [2]:
num_nodes = 10

In [3]:
def model(num_nodes: int, study: Study) -> do_mpc.model.Model:
    '''
        Create a model for a fixed bed adsorption column.
        The model includes advection but not dispersion of the fluid phase.
    '''

    dx = study.column_parameters.Length / num_nodes

    model = do_mpc.model.Model('continuous')

    C_Li = model.set_variable('_x', 'C_Li', shape=(num_nodes,1)) # liquid phase concentration in mol/m^3
    n_i = model.set_variable('_x', 'n_i', shape=(num_nodes,1)) # adsorbed phase concentration in mol/kg

    dn_i_dt = study.kinetics_experiments[0].kinetics_params.k2_si * (study.isotherm.isotherm_fit.q_eq_si(C_Li, study.kinetics_experiments[0].T) - n_i)**2
    model.set_rhs('n_i', dn_i_dt)

    C_up = vertcat(study.column_parameters.influent_concentration_si, C_Li[:-1])  # Upstream concentration with boundary condition
    dC_Li_dt = - study.column_parameters.interstitial_velocity_si[0] / dx * (C_Li - C_up) - (1 - study.column_parameters.porosity) / study.column_parameters.porosity * study.sorbent_properties.density * dn_i_dt

    model.set_rhs('C_Li', dC_Li_dt)
    model.setup()
    return model



In [4]:
s = Study.from_json('LiteratureReview/isotherm_kinetics.json', "jiangAdsorptionLithiumIons2020")

In [5]:
pprint(asdict(s), indent=2)

{ 'column_parameters': { 'Diameter': 0.02,
                         'Diameter_units': 'm',
                         'Length': 0.6,
                         'Length_units': 'm',
                         'flowrate': [2, 4, 8],
                         'flowrate_units': 'BV/h',
                         'influent_concentration': 350,
                         'influent_concentration_units': 'mg/L',
                         'porosity': 0.6},
  'isotherm': { 'T': 303,
                'equilibrium_concentration': [ 158.09523809523807,
                                               285.7142857142856,
                                               451.4285714285714,
                                               596.1904761904761,
                                               723.8095238095236,
                                               1194.2857142857142,
                                               1598.095238095238],
                'equilibrium_concentration_units': 'mg/L',
          

In [5]:
m = model(num_nodes, s)

In [ ]:
simulator = do_mpc.simulator.Simulator(m)
simulator.set_param(t_step=1.0)  # [s] time step for

simulator.setup()
simulator.reset_history()

simulator.x0['C_Li'] = np.zeros((num_nodes,1))
simulator.x0['n_i'] = np.zeros((num_nodes,1))

for i in tqdm(range(3*3600)):
    simulator.make_step()  # Advance the simulation by one time step
    

  0%|          | 0/10800 [00:00<?, ?it/s]CasADi - 2025-11-06 12:02:53 WARNING("simulator:daeF failed: Inf detected for output ode, at (row 0, col 0).") [.../casadi/core/oracle_function.cpp:408]
The right-hand side routine failed at the first call.
  0%|          | 0/10800 [00:00<?, ?it/s]


RuntimeError: Error in Function::call for 'simulator' [CvodesInterface] at .../casadi/core/function.cpp:1466:
Error in Function::call for 'simulator' [CvodesInterface] at .../casadi/core/function.cpp:362:
.../casadi/interfaces/sundials/cvodes_interface.cpp:399: CVode returned "CV_FIRST_RHSFUNC_ERR". Consult CVODES documentation.